In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
app_name_mapping = {
    'advocateoffice': 'Advocate Office',
    'automatedenrollment': 'Automated Enrollment',
    'bloodbanksystem': 'Blood Bank System',
    'clansphere': 'ClanSphere',
    'codeastro': 'CodeAstro',
    'collabtive': 'Collabtive',
    'covid19tms': 'Covid19tms',
    'dailyexpensetrackerproject': 'Daily Expense Tracker',
    'dairyfarmshop': 'Dairy Farm Management',
    'ecommercefruitsbazarmaster': 'Fruits Bazar',
    'engineersonlineportal': 'Engineers Online Portal',
    'fantasticblog': 'Fantastic Blog',
    'hospitalmanagementsystemproject': 'Hospital Management System',
    'keerti': 'Keerti',
    'loansystem': 'Loan Management',
    'seopanel': 'SEO Panel',
    'tailor': 'Tailor',
    'visitormanagementsystem': 'Visitor Management',
    'Sentinel': 'Sentinel',
    'jenkins-master': 'Jenkins',
    'forum-java': 'Forum Java',
    'obfuscator': 'Obfuscator',
    'scoold': 'Scoold',
    'tomcat55': 'Tomcat 5.5'
}

In [3]:
def get_metrics(file_name: str):
    df = pd.read_csv(f'../output/{file_name}')
    df['Approach'] = pd.Categorical(
        df['Approach'],
        categories=['Vanilla Joern', 'Vanilla Joern no DB', '+Dataflow', '+Sanitization', '+Database', '+Database no args'],
        ordered=True
    )
    df['Language'] = pd.Categorical(
        df['Language'],
        categories=['PHP', 'JAVASRC'],
        ordered=True
    )
    df = df.sort_values(by=['Language', 'Web Application', 'Approach'])
    df = df[['Web Application', 'Approach', 'Language', 'Code Injection Total Deduplicated Paths', 'Command Execution Total Deduplicated Paths', 'File Inclusion Total Deduplicated Paths', 'Session Fixation Total Deduplicated Paths', 'File Access Total Deduplicated Paths', 'SQL Injection Total Deduplicated Paths', 'XSS Total Deduplicated Paths']].rename(
        columns={
            'Code Injection Total Deduplicated Paths': 'Code Injection',
            'Command Execution Total Deduplicated Paths': 'Command Execution',
            'File Inclusion Total Deduplicated Paths': 'File Inclusion',
            'Session Fixation Total Deduplicated Paths': 'Session Fixation',
            'File Access Total Deduplicated Paths': 'File Access',
            'SQL Injection Total Deduplicated Paths': 'SQL Injection',
            'XSS Total Deduplicated Paths': 'XSS'
        }
    )
    return df

In [4]:
def generate_database_approach_table(df):
    # Filter data for +Database approach only
    db_data = df[df['Approach'] == '+Database'].copy()
    
    # Get unique web applications and sort them by language
    web_apps = sorted(db_data['Web Application'].unique())
    
    # Define vulnerability types
    vuln_types = ['Code Injection', 'Command Execution', 'File Inclusion', 
                  'Session Fixation', 'File Access', 'SQL Injection', 'XSS']
    
    latex_table = """\\begin{table*}[h!]
\\centering
\\scalebox{0.8}{
\\begin{tabular}{l""" + "c" * len(vuln_types) + """}
\\toprule
Web Application & """ + " & ".join(vuln_types) + """ \\\\
\\midrule
"""

    # Process PHP applications
    for app in web_apps:
        app_data = db_data[db_data['Web Application'] == app]
        display_name = app_name_mapping.get(app, app)
        
        if not app_data.empty:
            row = display_name
            for vuln in vuln_types:
                value = app_data[vuln].iloc[0]
                row += f" & {int(value)}"
            row += " \\\\\n"
            latex_table += row
    
    latex_table += """\\bottomrule
\\end{tabular}}
\\caption{Vulnerability detection results of \sysname{} across PHP applications on all covered vulnerabilities.}
\\label{tab:database_approach}
\\end{table*}"""
    
    return latex_table

In [5]:
def generate_ablation_latex_table(df):
    # Get unique web applications and sort them by language, then by name
    web_apps = sorted(df['Web Application'].unique())
    
    # Define the approaches for SQL Injection (no +Database) and XSS (with +Database)
    sql_approaches = ['Vanilla Joern', '+Dataflow', '+Sanitization']
    xss_approaches = ['Vanilla Joern', '+Dataflow', '+Sanitization', '+Database']
    
    latex_table = """\\begin{table*}[h!]
\\centering
\\scalebox{0.8}{
\\begin{tabular}{llcccccccc}
\\toprule
 &  & \\multicolumn{3}{c}{SQL Injection Vulnerabilities} & \\multicolumn{4}{c}{XSS Vulnerabilities} \\\\
\\cmidrule(lr){3-5} \\cmidrule(lr){6-9}
Language & App Name & """ + " & ".join(sql_approaches) + """ & """ + " & ".join(xss_approaches) + """ \\\\
\\midrule
"""
    
    # Process PHP applications first
    for i, app in enumerate(web_apps):
        app_data = df[df['Web Application'] == app]
        display_name = app_name_mapping.get(app, app)
        
        # Get SQL Injection values (3 columns)
        sql_values = []
        for approach in sql_approaches:
            approach_data = app_data[app_data['Approach'] == approach]
            if not approach_data.empty:
                value = approach_data['SQL Injection'].iloc[0]
                sql_values.append(str(int(value)))
            else:
                sql_values.append('0')
        
        # Get XSS values (4 columns)
        xss_values = []
        for approach in xss_approaches:
            approach_data = app_data[app_data['Approach'] == approach]
            if not approach_data.empty:
                value = approach_data['XSS'].iloc[0]
                xss_values.append(str(int(value)))
            else:
                xss_values.append('0')
        
        # Create the row - only show "PHP" for the first PHP app
        if i == 0:
            language_cell = f"\\multirow{{{len(web_apps)}}}{{*}}{{PHP}}"
        else:
            language_cell = ""
        
        row = f"{language_cell} & {display_name} & {' & '.join(sql_values)} & {' & '.join(xss_values)} \\\\\n"
        latex_table += row
    
    latex_table += """\\bottomrule
\\end{tabular}}
\\caption{Ablation study of SQL Injection and XSS total paths across different components of \\sysname{}.}
\\label{tab:ablation}
\\end{table*}"""
    
    return latex_table

In [6]:
df = get_metrics('all_stats.csv')
database_table = generate_database_approach_table(df)
print(database_table)

\begin{table*}[h!]
\centering
\scalebox{0.8}{
\begin{tabular}{lccccccc}
\toprule
Web Application & Code Injection & Command Execution & File Inclusion & Session Fixation & File Access & SQL Injection & XSS \\
\midrule
Advocate Office & 0 & 0 & 0 & 0 & 0 & 142 & 443 \\
Automated Enrollment & 0 & 0 & 0 & 0 & 0 & 537 & 4311 \\
Blood Bank System & 0 & 0 & 0 & 0 & 0 & 127 & 36 \\
ClanSphere & 4 & 0 & 9 & 440 & 454 & 306 & 1287 \\
CodeAstro & 0 & 0 & 0 & 0 & 0 & 73 & 384 \\
Collabtive & 3 & 0 & 0 & 0 & 172 & 0 & 1556 \\
Covid19tms & 0 & 0 & 0 & 0 & 0 & 63 & 7 \\
Daily Expense Tracker & 0 & 0 & 0 & 0 & 0 & 26 & 8 \\
Dairy Farm Management & 0 & 0 & 0 & 0 & 0 & 75 & 95 \\
Fruits Bazar & 0 & 20 & 0 & 0 & 0 & 263 & 298 \\
Engineers Online Portal & 0 & 0 & 0 & 0 & 0 & 466 & 4373 \\
Fantastic Blog & 0 & 0 & 0 & 0 & 2 & 22 & 9428 \\
Hospital Management System & 0 & 0 & 0 & 0 & 18 & 151 & 695 \\
Keerti & 0 & 0 & 0 & 0 & 0 & 16 & 0 \\
Loan Management & 0 & 0 & 0 & 0 & 0 & 52 & 62 \\
SEO Panel & 0 & 0 

In [7]:
latex_table = generate_ablation_latex_table(get_metrics('stats_ablation_study.csv'))
print(latex_table)

\begin{table*}[h!]
\centering
\scalebox{0.8}{
\begin{tabular}{llcccccccc}
\toprule
 &  & \multicolumn{3}{c}{SQL Injection Vulnerabilities} & \multicolumn{4}{c}{XSS Vulnerabilities} \\
\cmidrule(lr){3-5} \cmidrule(lr){6-9}
Language & App Name & Vanilla Joern & +Dataflow & +Sanitization & Vanilla Joern & +Dataflow & +Sanitization & +Database \\
\midrule
\multirow{7}{*}{PHP} & Advocate Office & 23 & 526 & 142 & 16 & 99 & 119 & 443 \\
 & Blood Bank System & 34 & 261 & 127 & 1 & 0 & 0 & 36 \\
 & CodeAstro & 24 & 234 & 73 & 21 & 11 & 11 & 384 \\
 & Collabtive & 42 & 11312 & 0 & 9 & 2403 & 1556 & 1556 \\
 & Fruits Bazar & 17 & 568 & 263 & 18 & 252 & 145 & 298 \\
 & Engineers Online Portal & 132 & 1165 & 466 & 67 & 715 & 137 & 4373 \\
 & Tailor & 48 & 432 & 97 & 21 & 102 & 76 & 1021 \\
\bottomrule
\end{tabular}}
\caption{Ablation study of SQL Injection and XSS total paths across different components of \sysname{}.}
\label{tab:ablation}
\end{table*}
